# Qwen2-VL LoRA Rank Sweep

Use this notebook to try LoRA rank/settings one run at a time, starting from the settings that already looked good:

- `shared_doc_text_lora`: rank 4, alpha 8, `q_proj,v_proj`.
- `chart_lora`: rank 8, alpha 16, `q_proj,k_proj,v_proj,o_proj`.

The goal is to find the best adapter settings before training a router. Do not run a huge full grid first; run the prioritized configs, compare validation metrics, then expand only around the winners.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CHART_SCRIPT = PROJECT_ROOT / "scripts/train_qwen2vl_chart_lora.py"
PREDICTIONS_DIR = PROJECT_ROOT / "outputs/predictions/rank_sweep"
CHECKPOINT_DIR = PROJECT_ROOT / "outputs/checkpoints/qwen2vl/rank_sweep"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

assert CHART_SCRIPT.exists(), f"Missing {CHART_SCRIPT}"
PROJECT_ROOT

## 2. Prioritized ChartQA Configs

Run these in order. The first config is the old good/default setting from the local ChartQA run. Only expand to stronger configs if `chart_hybrid_accuracy` improves or errors still look capacity-limited.

In [ ]:
CHART_CONFIGS = [
    {
        "name": "chart_r8_a16_B_lr2e-5_old_good",
        "lora_r": 8,
        "lora_alpha": 16,
        "learning_rate": "2e-5",
        "target_modules_preset": "B",
    },
    {
        "name": "chart_r8_a16_B_lr5e-5",
        "lora_r": 8,
        "lora_alpha": 16,
        "learning_rate": "5e-5",
        "target_modules_preset": "B",
    },
    {
        "name": "chart_r16_a32_B_lr2e-5",
        "lora_r": 16,
        "lora_alpha": 32,
        "learning_rate": "2e-5",
        "target_modules_preset": "B",
    },
    {
        "name": "chart_r16_a32_C_lr1e-5",
        "lora_r": 16,
        "lora_alpha": 32,
        "learning_rate": "1e-5",
        "target_modules_preset": "C",
    },
]

SAMPLE_LIMIT = 500
TRAIN_SIZE = 400
TEST_SIZE = 100
EPOCHS = 1
BATCH_SIZE = 1
GRAD_ACCUM = 4
SEED = 42
CHART_CONFIGS

## 3. Run ChartQA Sweep

Set `CONFIG_INDEX` and run one config at a time. Keep `RUN_ALL=False` unless you are ready to spend the time.

In [ ]:
CONFIG_INDEX = 0
RUN_ALL = False
PREPARE_IF_MISSING = True
STREAMING = True


def chart_command(config: dict) -> list[str]:
    output_dir = CHECKPOINT_DIR / config["name"]
    predictions_path = PREDICTIONS_DIR / f"{config['name']}_quality.jsonl"
    report_path = PREDICTIONS_DIR / f"{config['name']}_report.json"
    command = [
        sys.executable,
        str(CHART_SCRIPT),
        "--sample-limit",
        str(SAMPLE_LIMIT),
        "--train-size",
        str(TRAIN_SIZE),
        "--test-size",
        str(TEST_SIZE),
        "--epochs",
        str(EPOCHS),
        "--batch-size",
        str(BATCH_SIZE),
        "--gradient-accumulation-steps",
        str(GRAD_ACCUM),
        "--seed",
        str(SEED),
        "--learning-rate",
        str(config["learning_rate"]),
        "--lora-r",
        str(config["lora_r"]),
        "--lora-alpha",
        str(config["lora_alpha"]),
        "--target-modules-preset",
        config["target_modules_preset"],
        "--output-dir",
        str(output_dir.relative_to(PROJECT_ROOT)),
        "--predictions-path",
        str(predictions_path.relative_to(PROJECT_ROOT)),
        "--report-path",
        str(report_path.relative_to(PROJECT_ROOT)),
    ]
    if PREPARE_IF_MISSING:
        command.append("--prepare-if-missing")
    if STREAMING:
        command.append("--streaming")
    return command


configs_to_run = CHART_CONFIGS if RUN_ALL else [CHART_CONFIGS[CONFIG_INDEX]]
for config in configs_to_run:
    command = chart_command(config)
    print("Running:", " ".join(command))
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)

## 4. Compare ChartQA Reports

In [ ]:
def load_chart_report(config: dict) -> dict | None:
    report_path = PREDICTIONS_DIR / f"{config['name']}_report.json"
    if not report_path.exists():
        return None
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    return payload.get("chart_lora_local") or payload


rows = []
for config in CHART_CONFIGS:
    report = load_chart_report(config)
    if not report:
        continue
    overall = report["overall"]
    rows.append({
        "name": config["name"],
        "rank": config["lora_r"],
        "alpha": config["lora_alpha"],
        "lr": config["learning_rate"],
        "preset": config["target_modules_preset"],
        "chart_hybrid": overall.get("chart_hybrid_accuracy"),
        "norm_em": overall.get("normalized_exact_match"),
        "relaxed_num": overall.get("chart_relaxed_accuracy"),
        "avg_len": overall.get("avg_output_token_length"),
    })

rows = sorted(rows, key=lambda row: (row["chart_hybrid"] or 0, row["norm_em"] or 0), reverse=True)
print("name                                  rank alpha lr      preset hybrid  em     relaxed avg_len")
for row in rows:
    print(
        f"{row['name']:<37} {row['rank']:<4} {row['alpha']:<5} {row['lr']:<7} "
        f"{row['preset']:<6} {row['chart_hybrid']:.4f}  {row['norm_em']:.4f}  "
        f"{row['relaxed_num']:.4f}  {row['avg_len']:.2f}"
    )

best_chart_config = rows[0] if rows else None
best_chart_config

## 5. Shared Doc/Text And Hybrid Settings To Try

These configs are for the main notebook `train_qwen2vl_lora_baseline.ipynb`. Start from the old good config first. Train shared doc/text, train chart, then run hybrid eval.

In [ ]:
SHARED_DOC_TEXT_CONFIGS = [
    {
        "name": "shared_doc_text_r4_a8_A_lr1e-5_old_good",
        "lora_r": 4,
        "lora_alpha": 8,
        "learning_rate": "1e-5",
        "target_modules": ["q_proj", "v_proj"],
    },
    {
        "name": "shared_doc_text_r8_a16_A_lr1e-5",
        "lora_r": 8,
        "lora_alpha": 16,
        "learning_rate": "1e-5",
        "target_modules": ["q_proj", "v_proj"],
    },
    {
        "name": "shared_doc_text_r8_a16_B_lr1e-5",
        "lora_r": 8,
        "lora_alpha": 16,
        "learning_rate": "1e-5",
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    },
]

for config in SHARED_DOC_TEXT_CONFIGS:
    print(config)

## 6. Main Notebook Config Snippets

Copy one config at a time into the config cell of `train_qwen2vl_lora_baseline.ipynb`.

In [ ]:
def print_shared_doc_text_snippet(config: dict) -> None:
    print("ADAPTER_MODE = \"shared_doc_text_lora\"")
    print("TRAIN_ADAPTER = True")
    print("TRAIN_TASKS = None")
    print("ADAPTER_NAME = None")
    print(f"LORA_R = {config['lora_r']}")
    print(f"LORA_ALPHA = {config['lora_alpha']}")
    print(f"LEARNING_RATE = {config['learning_rate']}")
    print(f"LORA_TARGET_MODULES = {config['target_modules']!r}")


def print_hybrid_eval_snippet(use_base_chart_fallback: bool = False) -> None:
    print("ADAPTER_MODE = \"hybrid\"")
    print("TRAIN_ADAPTER = False")
    print(f"USE_BASE_FOR_CHARTQA_FALLBACK = {use_base_chart_fallback}")


print("# Shared doc/text old-good snippet")
print_shared_doc_text_snippet(SHARED_DOC_TEXT_CONFIGS[0])
print("\n# Hybrid eval snippet")
print_hybrid_eval_snippet(use_base_chart_fallback=False)

## 7. Compare Existing Hybrid Reports

After running the main notebook modes, this cell reads their reports and recommends the best backend per task.

In [ ]:
REPORTS = {
    "zero_shot": PROJECT_ROOT / "outputs/predictions/qwen2vl_zero_shot_report.json",
    "shared_lora_all_tasks": PROJECT_ROOT / "outputs/predictions/qwen2vl_shared_lora_all_tasks_report.json",
    "shared_doc_text_lora": PROJECT_ROOT / "outputs/predictions/qwen2vl_shared_doc_text_lora_report.json",
    "chart_lora_only": PROJECT_ROOT / "outputs/predictions/qwen2vl_chart_lora_only_report.json",
    "hybrid_chart_doc_text": PROJECT_ROOT / "outputs/predictions/qwen2vl_hybrid_chart_doc_text_report.json",
}


def load_method_report(method: str, path: Path) -> dict | None:
    if not path.exists():
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    return payload.get(method) or payload


def task_score(report: dict, task: str) -> float:
    task_report = report.get("by_task", {}).get(task, {})
    if task == "chartqa":
        return task_report.get("chart_hybrid_accuracy") or 0.0
    if task == "textvqa":
        return task_report.get("textvqa_vqa_score") or task_report.get("normalized_exact_match") or 0.0
    return task_report.get("normalized_exact_match") or 0.0


loaded = {method: load_method_report(method, path) for method, path in REPORTS.items()}
loaded = {method: report for method, report in loaded.items() if report}

print("Method                    Overall  ChartQA  DocVQA  TextVQA  AvgLen")
for method, report in loaded.items():
    overall = report["overall"]
    print(
        f"{method:<25} "
        f"{overall.get('normalized_exact_match', 0):.4f}  "
        f"{task_score(report, 'chartqa'):.4f}  "
        f"{task_score(report, 'docvqa'):.4f}  "
        f"{task_score(report, 'textvqa'):.4f}  "
        f"{overall.get('avg_output_token_length', 0):.2f}"
    )

print("\nBest backend by task:")
for task in ["chartqa", "docvqa", "textvqa"]:
    ranked = sorted(
        [(method, task_score(report, task)) for method, report in loaded.items()],
        key=lambda item: item[1],
        reverse=True,
    )
    if ranked:
        print(f"{task}: {ranked[0][0]} ({ranked[0][1]:.4f})")